# 🗄️ **Proyecto Júpiter: FakeNews - Notebook 2: Ingesta Automatizada en MySQL y Carga Relacional**

> **Consolidación de la persistencia relacional.** Este cuaderno toma como punto de partida el ecosistema de archivos estructurados en la **Tercera Forma Normal (3FN)** almacenados en la carpeta local `datos_3FN/` (en formato CSV) y formaliza la fase de **Ingeniería de Datos y ETL/ELT**. Su objetivo principal es unificar y automatizar el pipeline de creación de esquemas y carga masiva de datos dentro de un motor relacional **MySQL**, secuenciando la inyección de forma estrictamente ordenada para respetar la jerarquía de dependencias y garantizar la **integridad referencial** del modelo.

---

### 📋 **Ficha Técnica del Pipeline de Base de Datos**

| Campo | Detalle Técnico del Entorno |
|---|---|
| **Autores** | AB, JE, IF & JR - Proyecto Júpiter: Fake News (PontIA) |
| **Naturaleza del problema** | Ingeniería de datos: automatización de la creación de tablas y la carga de datos en una base relacional. |
| **Origen de los datos** | Capa limpia y atomizada en Tercera Forma Normal (`datos_3FN/` como ficheros CSV). |
| **Motor de Destino** | Servidor local **MySQL Server** (infraestructura unificada del equipo). |
| **Control de calidad** | Carga en orden respetando las relaciones entre tablas (claves foráneas), para que no queden datos sueltos ni inconsistentes. |
| **Resultado** | Base de datos relacional poblada e íntegra, consultable por SQL. |

---

### 🗺️ **Estructura del Pipeline de Ingesta (Los 4 Pilares Técnicos)**

Para asegurar una carga limpia y atómica que impida fallos por violación de restricciones del motor relacional, la arquitectura del código se divide en cuatro bloques esenciales:

1. **Conexión Local y Configuración del Driver:**
   * **Conectividad Robusta:** Inicialización del driver de comunicación (`mysql-connector-python` o el motor relacional de `SQLAlchemy`) apuntando a la instancia local de MySQL.
   * **Gestión Segura de Credenciales:** Configuración limpia de variables de entorno o parámetros de conexión centralizados para garantizar el acceso hermético al servidor de bases de datos.
2. **Secuenciación de la Jerarquía de Claves Foráneas (Anti-Lock Error):**
   * El pipeline implementa un diseño ordenado en cascada. Para evitar que MySQL lance un error crítico de restricción de integridad (`Foreign Key Constraint Violation`), el script discrimina el orden de ejecución:
     * **Fase A (Tablas Padre / Dimensiones Independientes):** Creación e inyección prioritaria de tablas que no dependen de nadie (las que no contienen claves ajenas en su estructura).
     * **Fase B (Tablas Hijo / Tablas de Hechos o Puentes):** Creación e inyección secuencial de las entidades complejas que referencian a las tablas padre mediante claves foráneas.
3. **Motor de Carga Masiva y Deserialización Atómica:**
   * **Lectura desde `datos_3FN/`:** Iteración automatizada sobre los archivos CSV limpios del proyecto.
   * **Mapeo de Tipos de Datos SQL:** Conversión estricta de tipos de datos de Pandas a formatos nativos de MySQL (`INT`, `VARCHAR`, `TEXT`, `DATETIME`, etc.), blindando la consistencia del esquema.
   * **Transaccionalidad Failsafe:** Inyección mediante bloques protegidos. En caso de que un solo registro falle, el motor ejecuta un `ROLLBACK` para evitar dejar la base de datos en un estado corrupto o a medio rellenar.
4. **Auditoría Forense de Carga (Lineage Logs):**
   * Generación y visualización de un reporte ejecutivo final que detalla el conteo exacto de registros insertados con éxito por cada tabla.
   * Registro analítico de tiempos de ejecución de las queries para asegurar la eficiencia del proceso de ingesta.

---

### 🏁 **Garantías de Salida (El Contrato de Datos Relacional)**

Al finalizar la ejecución de este cuaderno, el proyecto consigue:
* **Estructura Limpia:** Mantener la carpeta raíz del proyecto libre de archivos sueltos, concentrando la materia prima en `./datos_3FN/`.
* **Fuente única para consultas SQL:** La base de datos queda como infraestructura consultable directamente en SQL, independiente del pipeline de EDA y modelado (que trabaja sobre los Parquet de `datos_1FN/`).

# **3. Pipeline de ingesta relacional: carga y mapeo de datos en MySQL**

## 3.1. Configuración del entorno e importación de dependencias base

In [1]:
import os                                           # Rutas y comprobación de archivos en disco
import re                                           # Expresiones regulares (limpieza/validación de cadenas)
import pandas as pd                                 # Lectura de los CSV y manejo de DataFrames
import numpy as np                                  # Soporte numérico para pandas
from sqlalchemy import create_engine, text          # Conexión a MySQL y ejecución de sentencias SQL
from pathlib import Path                            # Gestión universal de rutas
from getpass import getpass                         # Pedir la contraseña de MySQL sin mostrarla en pantalla

## 3.2. Enrutamiento dinámico y conexión al motor relacional

In [2]:
# 1. Definimos las rutas universales
RUTA_BASE = Path('.')
CARPETA_MYSQL = RUTA_BASE / 'datos_3FN'

# 2. CONFIGURACIÓN DE TU BASE DE DATOS
USUARIO   = 'root'          
HOST      = '127.0.0.1'
PUERTO    = '3306'
DB_NOMBRE = 'db_fake_news'

# 3. Cuadro de texto interactivo seguro
# Al ejecutar esta celda, se abrirá un cuadro flotante para que escribas tu clave en secreto
PASSWORD = getpass("🔑 Introduce tu contraseña de MySQL y pulsa Enter: ")

# 4. Motor global de SQLAlchemy
engine = create_engine(f"mysql+pymysql://{USUARIO}:{PASSWORD}@{HOST}:{PUERTO}/{DB_NOMBRE}")

print(f"✅ Entorno configurado y motor relacional conectado de forma segura.")

✅ Entorno configurado y motor relacional conectado de forma segura.


## 3.3. Inserción de dimensiones puras independientes (dim_tipo y dim_pais)

### 3.3.1. Bloque: INSERCION_SQL_TIPO

In [3]:
TABLA_DESTINO = 'dim_tipo'
ruta_csv = CARPETA_MYSQL / 'tipo_3FN.csv'

try:
    # 1. Leer el archivo usando la nueva ruta dinámica
    df = pd.read_csv(ruta_csv)

    # --- PROCESAMIENTO DE DATOS ---
    COLUMNA_CSV = 'tipo' 
    
    if COLUMNA_CSV in df.columns:
        # Limpieza inicial: quitar nulos, duplicados y espacios extra
        tipos_unicos = df[[COLUMNA_CSV]].dropna().drop_duplicates()
        tipos_unicos[COLUMNA_CSV] = tipos_unicos[COLUMNA_CSV].str.strip()
    else:
        raise Exception(f"No se encontró la columna '{COLUMNA_CSV}' en el CSV")

    # --- FILTRAR CONTRA BASE DE DATOS ---
    # Usamos el 'engine' global que ya creamos en la celda 4.2
    with engine.connect() as conn:
        query = text(f"SELECT TIPO FROM {TABLA_DESTINO}")
        try:
            tipos_en_db = pd.read_sql(query, conn)['TIPO'].tolist()
        except:
            tipos_en_db = []
        
        # Filtramos el DataFrame: nos quedamos con lo que NO está en la DB
        df_para_insertar = tipos_unicos[~tipos_unicos[COLUMNA_CSV].isin(tipos_en_db)]

    # 3. Inserción
    if not df_para_insertar.empty:
        print(f"Insertando {len(df_para_insertar)} tipos de artículos nuevos...")
        df_para_insertar.columns = ['TIPO']
        
        df_para_insertar.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)
        print("✅ ¡Inserción de tipos completada con éxito!")
    else:
        print("ℹ Todos los tipos ya existen en la base de datos.")

except Exception as e:
    print(f"\n❌ ERROR DETECTADO: {e}")

Insertando 21 tipos de artículos nuevos...
✅ ¡Inserción de tipos completada con éxito!


### 3.3.2. Bloque: INSERCION_SQL_PAIS_IDIOMA

In [4]:
TABLA_DESTINO = 'dim_pais'
ruta_csv = CARPETA_MYSQL / 'pais_idioma_3FN.csv'

try:
    # 1. Leer el archivo usando la nueva ruta dinámica
    df = pd.read_csv(ruta_csv)

    # --- PROCESAMIENTO DE DATOS ---
    COL_CSV = 'pais' 
    COL_SQL = 'PAIS'

    if COL_CSV in df.columns:
        # Extraemos países únicos, limpiamos espacios y quitamos nulos
        paises_unicos = df[[COL_CSV]].dropna().drop_duplicates()
        paises_unicos[COL_CSV] = paises_unicos[COL_CSV].str.strip()
    else:
        raise Exception(f"No se encontró la columna '{COL_CSV}'. Columnas disponibles: {df.columns.tolist()}")

    # --- FILTRAR CONTRA BASE DE DATOS ---
    print(f"Verificando países ya existentes en {TABLA_DESTINO}...")
    with engine.connect() as conn:
        query = text(f"SELECT {COL_SQL} FROM {TABLA_DESTINO}")
        # Manejamos el caso de que la tabla esté vacía
        try:
            paises_en_db = pd.read_sql(query, conn)[COL_SQL].tolist()
        except:
            paises_en_db = []
        
        # Filtramos para insertar solo lo nuevo
        df_para_insertar = paises_unicos[~paises_unicos[COL_CSV].isin(paises_en_db)].copy()

    # 3. Inserción
    if not df_para_insertar.empty:
        # Renombramos para que coincida con el nombre de la columna en SQL
        df_para_insertar.columns = [COL_SQL]
        
        print(f"Insertando {len(df_para_insertar)} países nuevos...")
        # MySQL generará el PAIS_ID automáticamente
        df_para_insertar.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)
        print("✅ ¡Inserción de países completada con éxito!")
    else:
        print("ℹ No hay países nuevos para agregar.")

except Exception as e:
    print("\n❌ ERROR DETECTADO:")
    if hasattr(e, 'orig'):
        print(f"Código MySQL: {e.orig.args[0]} - {e.orig.args[1]}")
    else:
        print(f"Detalle: {e}")

Verificando países ya existentes en dim_pais...
Insertando 35 países nuevos...
✅ ¡Inserción de países completada con éxito!


## 3.4. Inserción de dimensiones operacionales limpias (DIM_AUTOR y DIM_FUENTE)

### 3.4.1. Bloque: INSERCION_SQL_AUTOR

In [5]:
TABLA_DESTINO = 'DIM_AUTOR'
ruta_csv = CARPETA_MYSQL / 'info_3FN.csv'

try:
    df = pd.read_csv(ruta_csv)
    
    if 'autor' in df.columns:
        autores_unicos = df[['autor']].dropna().drop_duplicates()
        autores_unicos['autor'] = autores_unicos['autor'].str.strip()

        # Limpieza de autores fantasmas que contienen números
        autores_unicos['autor'] = autores_unicos['autor'].apply(
            lambda x: np.nan if re.search(r'\d', str(x)) else x
        )
        autores_unicos = autores_unicos.dropna(subset=['autor'])
    else:
        raise Exception("No se encontró la columna 'autor'")

    with engine.connect() as conn:
        query = text(f"SELECT AUTOR FROM {TABLA_DESTINO}")
        try:
            autores_en_db = pd.read_sql(query, conn)['AUTOR'].tolist()
        except:
            autores_en_db = []

        df_para_insertar = autores_unicos[~autores_unicos['autor'].isin(autores_en_db)].copy()

    if not df_para_insertar.empty:
        print(f"Insertando {len(df_para_insertar)} autores nuevos...")
        df_para_insertar.columns = ['AUTOR']
        df_para_insertar.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)
        print(f"✅ ¡Inserción en {TABLA_DESTINO} completada con éxito!")
    else:
        print(f"ℹ Todos los autores ya existen en {TABLA_DESTINO}.")

except Exception as e:
    print(f"❌ ERROR DETECTADO: {e}")

Insertando 34 autores nuevos...
✅ ¡Inserción en DIM_AUTOR completada con éxito!


C:\Users\<usuario>\AppData\Local\Temp\ipykernel_19856\612286900.py:31: UserWarning: The provided table name 'DIM_AUTOR' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_para_insertar.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)


### 3.4.2. Bloque: INSERCION_SQL_FUENTE

In [6]:
TABLA_DESTINO = 'DIM_FUENTE'
ruta_csv = CARPETA_MYSQL / 'info_3FN.csv'

try:
    df = pd.read_csv(ruta_csv)
    COL_CSV = 'fuente'
    COL_SQL = 'FUENTE'

    if COL_CSV in df.columns:
        fuentes_unicas = df[[COL_CSV]].dropna().drop_duplicates()
        fuentes_unicas[COL_CSV] = fuentes_unicas[COL_CSV].str.strip()
    else:
        raise Exception(f"No se encontró la columna '{COL_CSV}'")

    print(f"Verificando fuentes ya existentes en {TABLA_DESTINO}...")
    with engine.connect() as conn:
        query = text(f"SELECT {COL_SQL} FROM {TABLA_DESTINO}")
        try:
            fuentes_en_db = pd.read_sql(query, conn)[COL_SQL].tolist()
        except:
            fuentes_en_db = []
        
        df_para_insertar = fuentes_unicas[~fuentes_unicas[COL_CSV].isin(fuentes_en_db)].copy()

    if not df_para_insertar.empty:
        df_para_insertar.columns = [COL_SQL]
        print(f"Insertando {len(df_para_insertar)} fuentes nuevas...")
        df_para_insertar.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)
        print(f"✅ ¡Inserción en {TABLA_DESTINO} completada con éxito!")
    else:
        print(f"ℹ No hay fuentes nuevas que insertar en {TABLA_DESTINO}.")

except Exception as e:
    print(f"❌ ERROR DETECTADO: {e}")

Verificando fuentes ya existentes en DIM_FUENTE...
Insertando 33 fuentes nuevas...
✅ ¡Inserción en DIM_FUENTE completada con éxito!


C:\Users\<usuario>\AppData\Local\Temp\ipykernel_19856\2782517340.py:28: UserWarning: The provided table name 'DIM_FUENTE' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_para_insertar.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)


## 3.5. Consolidación del dataset de entrada y mapeo de integridad referencial (TB_HECHOS)

### 3.5.1. Bloque: INSERCION_SQL_HECHOS

In [7]:
TABLA_DESTINO = 'TB_HECHOS'

try:
    print(f"Consolidando archivos de la carpeta: {CARPETA_MYSQL}")
    csv_files = [f for f in os.listdir(CARPETA_MYSQL) if f.endswith('.csv')]

    df_maestro = pd.DataFrame()

    for file in csv_files:
        file_path = CARPETA_MYSQL / file
        df_temp = pd.read_csv(file_path)
        df_temp.columns = [c.lower().strip() for c in df_temp.columns]
        
        if df_maestro.empty:
            df_maestro = df_temp
        else:
            df_maestro = pd.merge(df_maestro, df_temp, on='t_id', how='outer', suffixes=('', '_drop'))

    df_maestro = df_maestro.loc[:, ~df_maestro.columns.str.contains('_drop')]
    print(f"Consolidación completada. Columnas encontradas: {df_maestro.columns.tolist()}")

    # Descargamos los mapas relacionales actuales usando las MAYÚSCULAS del SQL original
    with engine.connect() as conn:
        dim_autor = pd.read_sql("SELECT ID_AUTOR, AUTOR FROM DIM_AUTOR", conn)
        dim_fuente = pd.read_sql("SELECT ID_FUENTE, FUENTE FROM DIM_FUENTE", conn)
        dim_pais = pd.read_sql("SELECT ID_PAIS, PAIS FROM DIM_PAIS", conn)
        dim_tipo = pd.read_sql("SELECT ID_TIPO, TIPO FROM DIM_TIPO", conn)

    # Normalizamos los nombres de columnas que devuelve SQL para poder cruzarlos en Pandas
    for d in [dim_autor, dim_fuente, dim_pais, dim_tipo]:
        d.columns = d.columns.str.lower()

    # Mapeo de textos a IDs relacionales
    df_maestro = df_maestro.merge(dim_autor, on='autor', how='left')
    df_maestro = df_maestro.merge(dim_fuente, on='fuente', how='left')
    df_maestro = df_maestro.merge(dim_pais, on='pais', how='left')
    df_maestro = df_maestro.merge(dim_tipo, on='tipo', how='left')

    cols_sql = {
        't_id': 'T_ID',
        'titulo': 'TITULO',
        'id_tipo': 'ID_TIPO',
        'id_autor': 'ID_AUTOR',
        'id_fuente': 'ID_FUENTE',
        'id_pais': 'ID_PAIS',
        'fake': 'FAKE',
        'tiempo': 'TIEMPO',
        'compartir_tiempo': 'COMPARTIR_TIEMPO',
        'texto': 'TEXTO'
    }

    columnas_existentes = [c for c in cols_sql.keys() if c in df_maestro.columns]
    df_final = df_maestro[columnas_existentes].rename(columns=cols_sql)
    df_final = df_final.drop_duplicates(subset=['T_ID'])
    
    # Inserción final en la tabla de hechos en mayúsculas
    df_final.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)
    print(f"✅ ¡Éxito! Se han mapeado e insertado {len(df_final)} noticias en {TABLA_DESTINO}.")

except Exception as e:
    print(f"❌ ERROR CRÍTICO EN HECHOS: {e}")

Consolidando archivos de la carpeta: datos_3FN
Consolidación completada. Columnas encontradas: ['t_id', 'compartir_tiempo', 'texto', 'titulo', 'fake', 'autor', 'fuente', 'compartir', 'duracion', 'favorito', 'visitas', 'pais', 'idioma', 'tiempo', 'tipo']


C:\Users\<usuario>\AppData\Local\Temp\ipykernel_19856\1450782615.py:57: UserWarning: The provided table name 'TB_HECHOS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_final.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)


✅ ¡Éxito! Se han mapeado e insertado 6335 noticias en TB_HECHOS.


## 3.6. Carga final de registros técnicos correlacionados (tb_metadatos)

### 3.6.1. Bloque: INSERCION_SQL_METADATOS

In [8]:
TABLA_DESTINO = 'TB_METADATOS'
ruta_csv = CARPETA_MYSQL / 'metadatos_3FN.csv'

try:
    df = pd.read_csv(ruta_csv)
    df.columns = [c.upper().strip() for c in df.columns]
    nombre_col_id_sql = 'T_ID'

    # Corrección automática de negativos
    cols_numericas = df.select_dtypes(include=[np.number]).columns
    for col in cols_numericas:
        if col != nombre_col_id_sql:
            if (df[col] < 0).any():
                print(f"Corrigiendo valores negativos en: {col}")
                df[col] = df[col].abs()
            
    df_limpio = df.dropna(subset=[nombre_col_id_sql]).copy()
    df_limpio[nombre_col_id_sql] = df_limpio[nombre_col_id_sql].astype(int)
    df_limpio = df_limpio.drop_duplicates(subset=[nombre_col_id_sql])
    df_limpio = df_limpio.replace({np.nan: None})

    print(f"Verificando integridad referencial con TB_HECHOS...")
    with engine.connect() as conn:
        try:
            res_existentes = conn.execute(text(f"SELECT {nombre_col_id_sql} FROM {TABLA_DESTINO}"))
            ids_existentes = [row[0] for row in res_existentes]
        except:
            ids_existentes = []

        # Buscamos en TB_HECHOS (en mayúsculas) los IDs válidos
        res_hechos = conn.execute(text(f"SELECT T_ID FROM TB_HECHOS"))
        ids_en_hechos = [row[0] for row in res_hechos]

        # Filtramos: que existan en hechos y que no estén ya metidos en metadatos
        df_para_insertar = df_limpio[
            (~df_limpio[nombre_col_id_sql].isin(ids_existentes)) & 
            (df_limpio[nombre_col_id_sql].isin(ids_en_hechos))
        ]

    if not df_para_insertar.empty:
        print(f"Insertando {len(df_para_insertar)} registros válidos en {TABLA_DESTINO}...")
        df_para_insertar.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)
        print(f"✅ ¡Inserción en {TABLA_DESTINO} completada con éxito!")
    else:
        print(f"ℹ No hay registros nuevos válidos para insertar en {TABLA_DESTINO}.")

except Exception as e:
    print(f"❌ ERROR DETECTADO: {e}")




    # PASAR NEGATIVOS EN FAVORITO A NULOS

Corrigiendo valores negativos en: COMPARTIR
Corrigiendo valores negativos en: DURACION
Corrigiendo valores negativos en: FAVORITO
Corrigiendo valores negativos en: VISITAS
Verificando integridad referencial con TB_HECHOS...
Insertando 6335 registros válidos en TB_METADATOS...
✅ ¡Inserción en TB_METADATOS completada con éxito!


C:\Users\<usuario>\AppData\Local\Temp\ipykernel_19856\1152543132.py:42: UserWarning: The provided table name 'TB_METADATOS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_para_insertar.to_sql(name=TABLA_DESTINO, con=engine, if_exists='append', index=False)


## 3.7. Control de Calidad: Verificación de Carga Automatizada

In [9]:
print(" Iniciando auditoría automática de la base de datos relacional...")

# Lista de tablas en el orden exacto de tu script SQL
tablas_a_auditar = ['DIM_AUTOR', 'DIM_FUENTE', 'DIM_PAIS', 'DIM_TIPO', 'TB_HECHOS', 'TB_METADATOS']
reporte_auditoria = []

with engine.connect() as conn:
    for tabla in tablas_a_auditar:
        try:
            # Ejecutamos el conteo directo en MySQL
            resultado = conn.execute(text(f"SELECT COUNT(*) FROM {tabla}")).scalar()
            
            # Asignamos un estado visual rápido según el volumen
            if resultado == 0:
                estado = '❌ VACÍA (Incompleto)'
            elif tabla in ['TB_HECHOS', 'TB_METADATOS'] and resultado != 6335:
                estado = f'⚠️ VOLUMETRÍA ANÓMALA ({resultado}/6335)'
            else:
                estado = '✅ OK (Cargada)'
                
            reporte_auditoria.append({'Tabla': tabla, 'Registros': resultado, 'Estado': estado})
            
        except Exception as e:
            reporte_auditoria.append({'Tabla': tabla, 'Registros': 0, 'Estado': f'❌ ERROR CRÍTICO: {str(e)[:40]}...'})

# Convertimos el reporte en un DataFrame limpio para visualizarlo como una tabla analítica
df_reporte = pd.DataFrame(reporte_auditoria)

print("\n========================================================")
print("  REPORTE GLOBAL DE VERIFICACIÓN DE INGESTA (MySQL)   ")
print("========================================================")
print(df_reporte.to_string(index=False))
print("========================================================\n")

# Alerta de semáforo final para el usuario
filas_hechos = df_reporte.loc[df_reporte['Tabla'] == 'TB_HECHOS', 'Registros'].values[0]
if filas_hechos == 6335:
    print("✅ ¡PIPELINE DE INGESTA INTEGRAL COMPLETADO CON ÉXITO!")
    print("✅ Los datos relacionales están listos para la explotación de KPIs en SQL y el Notebook 3.")
else:
    print("❌ DETENCIÓN DEL PIPELINE: La base de datos no ha alcanzado la consistencia numérica requerida.")

 Iniciando auditoría automática de la base de datos relacional...

  REPORTE GLOBAL DE VERIFICACIÓN DE INGESTA (MySQL)   
       Tabla  Registros         Estado
   DIM_AUTOR         34 ✅ OK (Cargada)
  DIM_FUENTE         33 ✅ OK (Cargada)
    DIM_PAIS         35 ✅ OK (Cargada)
    DIM_TIPO         21 ✅ OK (Cargada)
   TB_HECHOS       6335 ✅ OK (Cargada)
TB_METADATOS       6335 ✅ OK (Cargada)

✅ ¡PIPELINE DE INGESTA INTEGRAL COMPLETADO CON ÉXITO!
✅ Los datos relacionales están listos para la explotación de KPIs en SQL y el Notebook 3.
